In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from CIT.evaluation.utils import load_jsonl

In [ ]:
answers_ft_folder="/./src/CIT/training/models/cv/data/test_answers/ft_02.06_with_at_least_1_url_eval_all_data/ft"
all_scores = []
for file in os.listdir(answers_ft_folder):
    if file.endswith(".jsonl"):
        answers = load_jsonl(os.path.join(answers_ft_folder, file))
        for answer in answers:
            precision = answer["precision"]
            recall = answer["recall"]
            complexity = len(answer["urls"])
            all_scores.append({
                "quality": answer["quality"],
                "precision": precision,
                "recall": recall,
                "complexity": complexity
            })
df = pd.DataFrame(all_scores)
df["correct_answer"] = df["quality"].apply(lambda x: 1 if x >=1 else 0)
df["perfect_answer"] = df["quality"].apply(lambda x: 1 if x == 2 else 0)
        

# Distribution of scores depending on complexity

In [ ]:
df.columns

In [ ]:
#bar plot with x being complexity and y being the proportion of correct answers
plt.figure(figsize=(10, 6))
sns.barplot(x="complexity", y="correct_answer", data=df, ci=None)
plt.title("Proportion of Correct Answers by Complexity")
plt.xlabel("Complexity (Number of URLs)")
plt.ylabel("Proportion of Correct Answers")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Round score if needed, e.g., to nearest 0.1
df['score_rounded'] = df['precision'].round(2)

# Count frequency of each score per complexity
score_counts = df.groupby(['score_rounded', 'complexity']).size().reset_index(name='count')

# Plot using seaborn
sns.catplot(
    data=score_counts,
    x='score_rounded', y='count', hue='complexity',
    kind='bar', height=6, aspect=1.5
)
plt.title('Score Frequencies by Complexity')
plt.xlabel('Precision')
plt.ylabel('Count')
plt.show()

In [ ]:
sns.histplot(data=df, x="precision", hue="complexity", kde=True, stat="density", element="step", common_norm=False)
plt.title('Score Distribution by Complexity')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

for c in sorted(df['complexity'].unique()):
    subset = df[df['complexity'] == c]
    sns.kdeplot(subset['precision'], label=f'Complexity {c}', bw_adjust=0.2)

plt.title('Precision Density by Complexity')
plt.xlabel('Score')
plt.ylabel('Density')
plt.legend()
plt.show()

# With different models

In [ ]:
answers_ft_folder_2="./src/CIT/training/models/cv/data/test_answers/ft_27.5_2epochs/ft"
answers_ft_folder_1="./src/CIT/training/models/cv/data/test_answers/ft_27.5_1epochs/ft"
answers_base="./src/CIT/training/models/cv/data/test_answers/base"
answers_base70="./src/CIT/training/models/cv/data/test_answers/base_70b"
answers_without_0urls="./src/CIT/training/models/cv/data/test_answers/ft_02.06_with_at_least_1_url_eval_all_data/ft"
def load_scores(folder):
    all_scores = []
    for file in os.listdir(folder):
        if file.endswith(".jsonl"):
            answers = load_jsonl(os.path.join(folder, file))
            if folder== answers_ft_folder_1:
                model = "ft_1epoch"
            elif folder == answers_ft_folder_2:
                model = "ft_2epochs"
            elif folder == answers_base:
                model = "base"
            elif folder == answers_base70:
                model = "base_70b"
            elif folder == answers_without_0urls:
                model = "ft_wo0"
            else:
                model = "unknown"
            for answer in answers:
                precision = answer["precision"]
                recall = answer["recall"]
                complexity = len(answer["urls"])
                all_scores.append({
                    "quality": answer["quality"],
                    "precision": precision,
                    "recall": recall,
                    "complexity": complexity,
                    "model": model,
                    "question_rephrased_id": answer["question_rephrased_id"]
                })
    return all_scores

scores_ft_1 = load_scores(answers_ft_folder_1)
scores_ft_2 = load_scores(answers_ft_folder_2)
scores_base = load_scores(answers_base)
scores_base70 = load_scores(answers_base70)
scores_wo0 = load_scores(answers_without_0urls)
all_scores = scores_base + scores_base70+ scores_wo0 +scores_ft_1 + scores_ft_2
df_all = pd.DataFrame(all_scores)
df_all["correct_answer"] = df_all["quality"].apply(lambda x: 1 if x >=1 else 0)
df_all["perfect_answer"] = df_all["quality"].apply(lambda x: 1 if x == 2 else 0)

In [ ]:
df_all.groupby("complexity")["question_rephrased_id"].nunique().sort_index().plot(kind='bar', figsize=(10, 6))
#plt.yscale('log')
plt.title("Number of Unique Questions by Complexity")
plt.xlabel("Complexity (Number of URLs)")
plt.xticks(rotation=0)
plt.ylabel("Number of Unique Questions")
plt.tight_layout()
plt.show()

In [ ]:
# Plotting the results, bar plot with x being complexity, y being the proportion of correct answers, and hue being the model
plt.figure(figsize=(10, 6))
sns.barplot(x="complexity", y="correct_answer", hue="model", data=df_all, errorbar=None)
plt.title("Proportion of Correct Answers by Complexity and Model")
plt.xlabel("Complexity (Number of URLs)")
plt.ylabel("Proportion of Correct Answers")
plt.xticks(rotation=45)
plt.legend(title='Model')
plt.tight_layout()
plt.show()

In [ ]:
scores_complexity_0 = df_all[df_all['complexity'] == 0]
scores_complexity_1 = df_all[df_all['complexity'] == 1]
scores_complexity_2 = df_all[df_all['complexity'] == 2]
scores_complexity_3 = df_all[df_all['complexity'] == 3]
bw_adjust = 0.3
#plot 4 precision density for each 4 complexity , depending on model
fig,ax=plt.subplots(2, 2, figsize=(12, 10))
sns.kdeplot(data=scores_complexity_0, x='precision', hue='model', ax=ax[0, 0], bw_adjust=bw_adjust)
sns.kdeplot(data=scores_complexity_1, x='precision', hue='model', ax=ax[0, 1], bw_adjust=bw_adjust)
sns.kdeplot(data=scores_complexity_2, x='precision', hue='model', ax=ax[1, 0], bw_adjust=bw_adjust)
sns.kdeplot(data=scores_complexity_3, x='precision', hue='model', ax=ax[1, 1], bw_adjust=bw_adjust)
ax[0, 0].set_title('Complexity 0')
ax[0, 1].set_title('Complexity 1')
ax[1, 0].set_title('Complexity 2')
ax[1, 1].set_title('Complexity 3')  

plt.tight_layout()
plt.show()



In [ ]:
def plot_score_distribution_by_model_and_complexity(df_all, score_key='precision'):
    scores_complexity_0 = df_all[df_all['complexity'] == 0]
    scores_complexity_1 = df_all[df_all['complexity'] == 1]
    scores_complexity_2 = df_all[df_all['complexity'] == 2]
    scores_complexity_3 = df_all[df_all['complexity'] == 3]
    fig, ax = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(f'{score_key}: Distribution by Complexity and Model \n With kernel density estimator (only to facilitate comparison)', fontsize=16)
    sns.histplot(data=scores_complexity_0, x=score_key, hue='model', kde=True, stat='count', element='step', common_norm=False, ax=ax[0, 0])
    sns.histplot(data=scores_complexity_1, x=score_key, hue='model', kde=True, stat='count', element='step', common_norm=False, ax=ax[0, 1])
    sns.histplot(data=scores_complexity_2, x=score_key, hue='model', kde=True, stat='count', element='step', common_norm=False, ax=ax[1, 0])    
    sns.histplot(data=scores_complexity_3, x=score_key, hue='model', kde=False, stat='count', element='step', common_norm=False, ax=ax[1, 1])
    ax[0, 0].set_title('Complexity 0')
    ax[0, 1].set_title('Complexity 1')
    ax[1, 0].set_title('Complexity 2')  
    ax[1, 1].set_title('Complexity 3')

    #keep legend only for the first subplot
    for a in ax.flat[1:]:
        a.legend_.remove()

    plt.tight_layout()
    plt.show()

In [ ]:
plot_score_distribution_by_model_and_complexity(df_all, score_key='recall')

In [ ]:
plot_score_distribution_by_model_and_complexity(df_all, score_key='precision')